In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
df = pd.read_excel("Berlin_wind_multiclass1.xlsx")

In [3]:
power = df["X50Hertz..MW."]
boundaries = power.quantile([0.2, 0.4, 0.6, 0.8]).values
conditions = [
    power <= boundaries[0],
    (power > boundaries[0]) & (power <= boundaries[1]),
    (power > boundaries[1]) & (power <= boundaries[2]),
    (power > boundaries[2]) & (power <= boundaries[3]),
    power > boundaries[3]
]
df["target"] = np.select(conditions, [0, 1, 2, 3, 4])
df["target"].value_counts().sort_index()

target
0    14026
1    14006
2    14021
3    14012
4    14015
Name: count, dtype: int64

In [4]:
X = df.drop(["X50Hertz..MW.", "target", "Year", "Minute"], axis=1)
y = df["target"]
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
imputer = SimpleImputer(strategy="median")
x_train_clean = imputer.fit_transform(x_train)
x_test_clean = imputer.transform(x_test)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_clean)
x_test_scaled = scaler.transform(x_test_clean)
x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [5]:
def createmodel():
    return nn.Sequential(
        nn.Linear(x_train_tensor.shape[1], 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 5)
    )

In [6]:
def train_model(model, X, y, epochs=20, batch_size=64, lr=0.001):
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    loss_criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0

        for xb, yb in loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = loss_criterion(outputs, yb)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * xb.size(0)
            correct += (outputs.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)

        print(f"Epoch {epoch + 1:03d} | Loss: {loss_sum / total:.4f} | Accuracy: {100 * correct / total:.2f}%")

    return model

In [7]:
def test_model(model, X, y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        predictions = outputs.argmax(dim=1).cpu().numpy()
    actual = y.cpu().numpy()
    print("Accuracy:", accuracy_score(actual, predictions) * 100)
    print("Confusion Matrix:")
    print(confusion_matrix(actual, predictions))
    print("Classification Report:")
    print(classification_report(actual, predictions, digits=4, zero_division=0))

In [8]:
centralized_model = createmodel()
centralized_model = train_model(centralized_model, x_train_tensor, y_train_tensor, epochs=100)

Epoch 001 | Loss: 0.9393 | Accuracy: 58.16%
Epoch 002 | Loss: 0.8290 | Accuracy: 63.21%
Epoch 003 | Loss: 0.7826 | Accuracy: 65.74%
Epoch 004 | Loss: 0.7439 | Accuracy: 67.47%
Epoch 005 | Loss: 0.7166 | Accuracy: 68.72%
Epoch 006 | Loss: 0.6906 | Accuracy: 69.75%
Epoch 007 | Loss: 0.6678 | Accuracy: 71.11%
Epoch 008 | Loss: 0.6490 | Accuracy: 71.83%
Epoch 009 | Loss: 0.6333 | Accuracy: 72.50%
Epoch 010 | Loss: 0.6172 | Accuracy: 73.41%
Epoch 011 | Loss: 0.6079 | Accuracy: 73.72%
Epoch 012 | Loss: 0.5945 | Accuracy: 74.34%
Epoch 013 | Loss: 0.5876 | Accuracy: 74.63%
Epoch 014 | Loss: 0.5736 | Accuracy: 75.52%
Epoch 015 | Loss: 0.5649 | Accuracy: 75.89%
Epoch 016 | Loss: 0.5568 | Accuracy: 76.35%
Epoch 017 | Loss: 0.5491 | Accuracy: 76.70%
Epoch 018 | Loss: 0.5413 | Accuracy: 77.14%
Epoch 019 | Loss: 0.5311 | Accuracy: 77.49%
Epoch 020 | Loss: 0.5288 | Accuracy: 77.78%
Epoch 021 | Loss: 0.5220 | Accuracy: 77.97%
Epoch 022 | Loss: 0.5160 | Accuracy: 78.17%
Epoch 023 | Loss: 0.5103 | Accur

In [9]:
test_model(centralized_model, x_test_tensor, y_test_tensor)

Accuracy: 88.96974885844749
Confusion Matrix:
[[2623  172    7    3    0]
 [ 341 2269  184    7    0]
 [  11  244 2340  208    1]
 [   0    6  148 2541  108]
 [   0    0    1  105 2697]]
Classification Report:
              precision    recall  f1-score   support

           0     0.8817    0.9351    0.9076      2805
           1     0.8432    0.8101    0.8263      2801
           2     0.8731    0.8345    0.8534      2804
           3     0.8872    0.9065    0.8968      2803
           4     0.9612    0.9622    0.9617      2803

    accuracy                         0.8897     14016
   macro avg     0.8893    0.8897    0.8891     14016
weighted avg     0.8893    0.8897    0.8892     14016



In [10]:
client_labels = [
    [2, 3, 4],
    [0, 1, 4],
    [0, 1, 2, 3]
]
client_data = []
for labels in client_labels:
    mask = torch.isin(y_train_tensor, torch.tensor(labels, dtype=torch.long))
    client_data.append((x_train_tensor[mask], y_train_tensor[mask]))
client_sizes = [len(y) for _, y in client_data]
client_sizes

[33638, 33638, 44852]

In [11]:
global_model = createmodel()
local_models = [createmodel(), createmodel(), createmodel()]

In [12]:
rounds = 10
local_epochs = 5

for round_num in range(rounds):
    global_weights = global_model.state_dict()
    client_weights = []

    for client_idx, (client_X, client_y) in enumerate(client_data):
        local_models[client_idx].load_state_dict(global_weights)
        local_models[client_idx] = train_model(
            local_models[client_idx],
            client_X,
            client_y,
            epochs=local_epochs
        )
        client_weights.append(copy.deepcopy(local_models[client_idx].state_dict()))

    averaged_weights = copy.deepcopy(client_weights[0])

    for key in averaged_weights:
        if torch.is_floating_point(averaged_weights[key]):
            averaged_weights[key] = sum(
                client_weights[i][key] * client_sizes[i]
                for i in range(len(client_weights))
            ) / sum(client_sizes)
        else:
            averaged_weights[key] = client_weights[0][key]

    global_model.load_state_dict(averaged_weights)
    print(f"Federated round {round_num + 1}/{rounds}")
    test_model(global_model, x_test_tensor, y_test_tensor)

Epoch 001 | Loss: 0.6579 | Accuracy: 69.23%
Epoch 002 | Loss: 0.5471 | Accuracy: 74.88%
Epoch 003 | Loss: 0.5180 | Accuracy: 76.55%
Epoch 004 | Loss: 0.4921 | Accuracy: 77.76%
Epoch 005 | Loss: 0.4705 | Accuracy: 79.10%
Epoch 001 | Loss: 0.4799 | Accuracy: 77.39%
Epoch 002 | Loss: 0.3855 | Accuracy: 82.00%
Epoch 003 | Loss: 0.3625 | Accuracy: 83.22%
Epoch 004 | Loss: 0.3498 | Accuracy: 83.77%
Epoch 005 | Loss: 0.3368 | Accuracy: 84.25%
Epoch 001 | Loss: 0.9834 | Accuracy: 55.56%
Epoch 002 | Loss: 0.8641 | Accuracy: 61.52%
Epoch 003 | Loss: 0.8159 | Accuracy: 64.05%
Epoch 004 | Loss: 0.7794 | Accuracy: 65.42%
Epoch 005 | Loss: 0.7506 | Accuracy: 67.11%
Federated round 1/10
Accuracy: 31.53538812785388
Confusion Matrix:
[[  24   76 2676   29    0]
 [   0  146 2475  180    0]
 [   0   77 1993  734    0]
 [   0   17  529 2257    0]
 [   0    7    6 2790    0]]
Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.0086    0.0170      2805


In [13]:
test_model(global_model, x_test_tensor, y_test_tensor)
torch.save(global_model.state_dict(), "federated_wind_mlp.pth")

Accuracy: 70.99743150684932
Confusion Matrix:
[[2454  330   21    0    0]
 [ 268 2242  287    4    0]
 [  36  657 1780  330    1]
 [   2   68  143 2574   16]
 [   0    1    0 1901  901]]
Classification Report:
              precision    recall  f1-score   support

           0     0.8891    0.8749    0.8819      2805
           1     0.6798    0.8004    0.7352      2801
           2     0.7978    0.6348    0.7071      2804
           3     0.5352    0.9183    0.6763      2803
           4     0.9815    0.3214    0.4843      2803

    accuracy                         0.7100     14016
   macro avg     0.7767    0.7100    0.6970     14016
weighted avg     0.7767    0.7100    0.6970     14016

